# P1 v6 — 2-Group Severity Adapters + Soft MLP Routing

## Architecture

**Training:**
- B2 backbone: fully frozen
- 2 LoRA adapters + 2 CTC heads, initialised from LibriSpeech CTC weights
  - Group 0 (`mild_ctrl`): trains on control + mild utterances
  - Group 1 (`mod_sev`):   trains on moderate + severe utterances
- LibriSpeech init: higher initial loss → stronger gradients → more adaptation

**Inference:**
```
Audio → B2 encoder (frozen)
    ↓                              ↓
adapter_0 → CTC head_0      Layer-6 → Severity MLP (pretrained, 4-class)
adapter_1 → CTC head_1           ↓
    ↓                    p = [p_ctrl, p_mild, p_mod, p_sev]
logits_0, logits_1               ↓
    ↓               w_0 = p_ctrl + p_mild   (mild_ctrl group weight)
    ↓               w_1 = p_mod  + p_sev    (mod_sev group weight)
    ↓
final_logits = w_0 × logits_0 + w_1 × logits_1
    ↓
KenLM beam search
```

In [1]:
import os, warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("pyctcdecode").setLevel(logging.ERROR)
os.environ["CUDA_VISIBLE_DEVICES"]      = "0"
os.environ["TOKENIZERS_PARALLELISM"]    = "false"
os.environ["HF_HOME"]                   = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"]         = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]              = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"]        = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]            = "/kaggle/temp/.cache"
for p in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"],
           os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"],
           os.environ["XDG_CACHE_HOME"]]:
    os.makedirs(p, exist_ok=True)
print("Env ready.")

Env ready.


In [ ]:
!pip install pyctcdecode==0.5.0
!pip install kenlm
!pip install numpy==1.26.4
!pip -q install -U transformers datasets evaluate jiwer soundfile huggingface_hub scikit-optimize
!pip -q install safetensors

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import evaluate
import json
import random
import sys
from itertools import groupby
from collections import Counter
from typing import Optional

from datasets import load_dataset
from transformers import WavLMForCTC, Wav2Vec2Processor
from pyctcdecode import build_ctcdecoder
from huggingface_hub import hf_hub_download

print("torch:", torch.__version__)
print("GPU:",   torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
GPU: True
GPU: Tesla T4


## Step 1 — Config

In [3]:
B2_PATH        = "/kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final"
MLP_PT         = "/kaggle/input/datasets/kavishk/severity-mlp/p2_severity_mlp.pt"

RANDOM_SEED    = 42
TEST_SPEAKERS  = {"F01", "F04", "FC01", "M05"}
MLP_VAL_FRAC   = 0.2
CONTROL_TARGET = 1500
MAX_AUDIO      = 240_000
NUM_SEV_CLASSES = 4
NUM_GROUPS      = 2

TORGO_SEV_TO_INT = {"control": 0, "mild": 1, "moderate": 2, "severe": 3}
INT_TO_SEV       = {v: k for k, v in TORGO_SEV_TO_INT.items()}
SEVERITY_MAPPING = {
    "F01": "severe",   "M01": "severe",   "M02": "severe",   "M04": "severe",
    "M05": "moderate", "F03": "moderate",
    "F04": "mild",     "M03": "mild",
    "FC01": "control", "FC02": "control", "FC03": "control",
    "MC01": "control", "MC02": "control", "MC03": "control", "MC04": "control",
}
GROUP_NAMES  = ["mild_ctrl", "mod_sev"]
SEV_TO_GROUP = {"control": 0, "mild": 0, "moderate": 1, "severe": 1}
BEAM_WIDTH   = 100
BATCH        = 8

print("Config ready.")
print("Group 0 (mild_ctrl): control + mild")
print("Group 1 (mod_sev):   moderate + severe")

Config ready.
Group 0 (mild_ctrl): control + mild
Group 1 (mod_sev):   moderate + severe


## Step 2 — Load TORGO, create splits

In [4]:
print("Loading TORGO...")
cache = os.environ["HF_DATASETS_CACHE"]
ds    = load_dataset("abnerh/TORGO-database", cache_dir=cache)
df    = ds["train"].to_pandas()
df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"]    = df["audio_path"].apply(lambda p: str(p).split("_")[0])
df["Category"]   = df["speaker"].map(SEVERITY_MAPPING)
df["Group"]      = df["Category"].map(SEV_TO_GROUP)

hf_full = ds["train"].add_column("speaker",  df["speaker"].tolist())
hf_full = hf_full.add_column("Category", df["Category"].tolist())
hf_full = hf_full.add_column("Group",    df["Group"].tolist())

test_mask  = df["speaker"].isin(TEST_SPEAKERS)
torgo_test = hf_full.select(df[test_mask].index.tolist())
torgo_test = torgo_test.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO, num_proc=1
)

train_pool_df = df[~test_mask].copy()
ctrl  = train_pool_df[train_pool_df["Category"] == "control"].sample(
    n=min(CONTROL_TARGET, (train_pool_df["Category"] == "control").sum()),
    random_state=RANDOM_SEED
)
other = train_pool_df[train_pool_df["Category"] != "control"]
train_pool_df = pd.concat([ctrl, other]).sample(frac=1, random_state=RANDOM_SEED)

random.seed(RANDOM_SEED)
group_train_idx = {0: [], 1: []}
val_idx         = []
for sev in ["control", "mild", "moderate", "severe"]:
    grp     = SEV_TO_GROUP[sev]
    sev_idx = train_pool_df[train_pool_df["Category"] == sev].index.tolist()
    random.shuffle(sev_idx)
    n_val   = max(1, int(len(sev_idx) * MLP_VAL_FRAC))
    val_idx.extend(sev_idx[:n_val])
    group_train_idx[grp].extend(sev_idx[n_val:])

group_train = {
    grp: hf_full.select(idxs).filter(
        lambda x: len(x["audio"]["array"]) <= MAX_AUDIO, num_proc=1
    )
    for grp, idxs in group_train_idx.items()
}
torgo_val = hf_full.select(val_idx).filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO, num_proc=1
)

print(f"Test: {len(torgo_test)}  Val: {len(torgo_val)}")
for g, ds_ in group_train.items():
    cats = dict(sorted(Counter(ds_["Category"]).items()))
    print(f"Group {g} ({GROUP_NAMES[g]}): {len(ds_)} utterances  {cats}")

Loading TORGO...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/370M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/397M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/356M [00:00<?, ?B/s]

...

Filter (num_proc=1):   0%|          | 0/1798 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/1848 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/2615 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/1115 [00:00<?, ? examples/s]

Test: 1786  Val: 1111
Group 0 (mild_ctrl): 1848 utterances  {'control': 1200, 'mild': 648}
Group 1 (mod_sev): 2586 utterances  {'moderate': 869, 'severe': 1717}


## Step 3 — Load B2 (frozen) and extract LibriSpeech CTC weights

In [5]:
from safetensors.torch import load_file

processor = Wav2Vec2Processor.from_pretrained(B2_PATH)
b2_model  = WavLMForCTC.from_pretrained(
    B2_PATH, ctc_loss_reduction="mean", ctc_zero_infinity=True
)
for param in b2_model.parameters():
    param.requires_grad = False
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
b2_model = b2_model.to(device)
b2_model.eval()
print(f"B2 loaded on {device} — fully frozen")

# ── Extract CTC weights from LibriSpeech model ──
LIBRISPEECH_DIR = "/kaggle/input/datasets/kavishk/b1-wavlm-ctc-librispeech/kaggle/working/b1_wavlm_ctc_librispeech"

libri_state = load_file(os.path.join(LIBRISPEECH_DIR, "model.safetensors"))

# Print all keys to confirm lm_head location
lm_keys = [k for k in libri_state.keys() if "lm_head" in k]
print(f"lm_head keys: {lm_keys}")

libri_ctc_weight = libri_state["lm_head.weight"].clone()
libri_ctc_bias   = libri_state["lm_head.bias"].clone() if "lm_head.bias" in libri_state else None
print(f"LibriSpeech CTC weight: {libri_ctc_weight.shape}")

vocab_size  = processor.tokenizer.vocab_size
hidden_size = b2_model.config.hidden_size  # 768

if libri_ctc_weight.shape[0] != vocab_size:
    print(f"Vocab mismatch: LibriSpeech={libri_ctc_weight.shape[0]} TORGO={vocab_size} — aligning...")
    min_v = min(libri_ctc_weight.shape[0], vocab_size)
    nw    = torch.randn(vocab_size, hidden_size) * 0.02
    nw[:min_v].copy_(libri_ctc_weight[:min_v])
    libri_ctc_weight = nw
    if libri_ctc_bias is not None:
        nb = torch.zeros(vocab_size)
        nb[:min_v].copy_(libri_ctc_bias[:min_v])
        libri_ctc_bias = nb

print(f"CTC weights ready: {libri_ctc_weight.shape}")

Loading weights:   0%|          | 0/250 [00:00<?, ?it/s]

B2 loaded on cuda — fully frozen
lm_head keys: ['lm_head.bias', 'lm_head.weight']
LibriSpeech CTC weight: torch.Size([32, 768])
Vocab mismatch: LibriSpeech=32 TORGO=30 — aligning...
CTC weights ready: torch.Size([30, 768])


In [6]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

def prepare_torgo(batch):
    """
    Same preprocessing as B2/P1 training.
    Uses processor feature extraction identically to training.
    """
    audio = batch["audio"]
    arr   = audio["array"]
    if len(arr) > MAX_AUDIO:
        arr = arr[:MAX_AUDIO]
    inputs = processor(arr, sampling_rate=audio["sampling_rate"])
    batch["input_values"]   = inputs.input_values[0]
    batch["labels"]         = processor.tokenizer(
        batch["transcription"].lower().strip()
    ).input_ids
    batch["severity_label"] = TORGO_SEV_TO_INT[batch["Category"]]
    return batch


@dataclass
class DataCollatorCTC:
    """Pads input_values and labels — identical to B2/P1 DataCollatorLoRA."""    
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_feats  = [{"input_values": f["input_values"]} for f in features]
        batch        = self.processor.feature_extractor.pad(
            input_feats, padding=self.padding, return_tensors="pt"
        )
        label_feats  = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_feats, padding=self.padding, return_tensors="pt"
        )
        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        return batch


data_collator = DataCollatorCTC(processor=processor)
print("prepare_torgo and data_collator ready.")


prepare_torgo and data_collator ready.


In [7]:
# Preprocess using same pipeline as B2 training
# Avoids ~6pp WER difference from re-processing raw audio at inference

print("Preprocessing torgo_test...")
TORGO_COLS = torgo_test.column_names
test_p = torgo_test.map(
    prepare_torgo,
    remove_columns=TORGO_COLS,
    num_proc=1,
    desc="Preprocessing TORGO test",
)
test_cats_filtered = list(torgo_test["Category"])
print(f"test_p: {len(test_p)} utterances")

print("\nPreprocessing torgo_val...")
VAL_COLS  = torgo_val.column_names
torgo_val_p = torgo_val.map(
    prepare_torgo,
    remove_columns=VAL_COLS,
    num_proc=1,
    desc="Preprocessing TORGO val",
)
val_cats_filtered = list(torgo_val["Category"])
print(f"torgo_val_p: {len(torgo_val_p)} utterances")

# Also preprocess group_train datasets for training
print("\nPreprocessing group_train...")
group_train_p = {}
for grp, ds_ in group_train.items():
    COLS = ds_.column_names
    group_train_p[grp] = ds_.map(
        prepare_torgo,
        remove_columns=COLS,
        num_proc=1,
        desc=f"Preprocessing group {grp}",
    )
    print(f"  Group {grp} ({GROUP_NAMES[grp]}): {len(group_train_p[grp])} utterances")


Preprocessing torgo_test...


Preprocessing TORGO test (num_proc=1):   0%|          | 0/1786 [00:00<?, ? examples/s]

test_p: 1786 utterances

Preprocessing torgo_val...


Preprocessing TORGO val (num_proc=1):   0%|          | 0/1111 [00:00<?, ? examples/s]

torgo_val_p: 1111 utterances

Preprocessing group_train...


Preprocessing group 0 (num_proc=1):   0%|          | 0/1848 [00:00<?, ? examples/s]

  Group 0 (mild_ctrl): 1848 utterances


Preprocessing group 1 (num_proc=1):   0%|          | 0/2586 [00:00<?, ? examples/s]

  Group 1 (mod_sev): 2586 utterances


## Step 4 — Build 2-group LoRA model

In [8]:
class LoRALinear(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.weight = linear.weight
        self.bias   = linear.bias
        self.weight.requires_grad = False
        if self.bias is not None: self.bias.requires_grad = False
        self.lora_A = nn.Parameter(torch.empty(rank, linear.in_features))
        self.lora_B = nn.Parameter(torch.zeros(linear.out_features, rank))
        self.scale  = alpha / rank
        nn.init.normal_(self.lora_A)

    def forward(self, x):
        return (F.linear(x, self.weight, self.bias)
                + F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scale)


def inject_lora(encoder, rank, alpha):
    PROJ_NAMES = ["out_proj", "gru_rel_pos_linear"]
    targets    = []
    for _, module in encoder.named_modules():
        for proj in PROJ_NAMES:
            child = getattr(module, proj, None)
            if child is None: continue
            if type(child).__name__ == "LoRALinear": continue
            if hasattr(child, "weight") and hasattr(child, "in_features"):
                targets.append((module, proj))
    for parent, proj in targets:
        setattr(parent, proj, LoRALinear(getattr(parent, proj), rank, alpha))
    print(f"  Replaced {len(targets)} linear layers")
    return encoder


class CTCHead(nn.Module):
    """
    Bottleneck CTC head: 768→256→32 (net) + 768→32 (residual skip).
    Residual init from B2's CTC weights — starts as B2's exact output.
    Net starts near zero — learns severity-specific corrections on top.
    At init: output = net(x) + residual(x) ≈ 0 + B2_CTC(x) = B2's output.
    """
    def __init__(self, hidden_size, vocab_size, intermediate=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, intermediate),  # 768→256
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(intermediate, vocab_size),   # 256→32
        )
        self.residual = nn.Linear(hidden_size, vocab_size, bias=True)

    def forward(self, x):
        return self.net(x) + self.residual(x)


class TwoGroupLoRAModel(nn.Module):
    """
    Frozen B2 backbone (bottom 8 layers) + partially unfrozen (top 4 layers)
    + 2 LoRA adapters (rank=32) + 2 multi-layer CTC heads init from B2.
    """
    def __init__(self, b2_model, rank=32, alpha=64):
        super().__init__()
        hidden = b2_model.config.hidden_size   # 768
        vocab  = len(processor.tokenizer.get_vocab())  # 32

        self.encoder = b2_model.wavlm

        # Freeze ALL backbone first
        for p in self.encoder.parameters(): p.requires_grad = False

        # Unfreeze top 4 transformer layers at low LR
        for i in range(8, 12):
            for p in self.encoder.encoder.layers[i].parameters():
                p.requires_grad = True
        print(f"  Top 4 backbone layers unfrozen (layers 8-11)")

        print("Injecting LoRA...")
        self.encoder      = inject_lora(self.encoder, rank, alpha)
        self._lora_layers = [
            m for m in self.encoder.modules()
            if type(m).__name__ == "LoRALinear"
        ]
        print(f"  {len(self._lora_layers)} LoRALinear layers")

        # 2 adapter parameter stores
        self.adapter_A = nn.ParameterList([
            nn.ParameterList([nn.Parameter(l.lora_A.data.clone())
                              for l in self._lora_layers])
            for _ in range(NUM_GROUPS)
        ])
        self.adapter_B = nn.ParameterList([
            nn.ParameterList([nn.Parameter(l.lora_B.data.clone())
                              for l in self._lora_layers])
            for _ in range(NUM_GROUPS)
        ])

        # Extract B2's CTC weights for head initialisation
        b2_ctc_w = b2_model.lm_head.weight.data.clone()  # [32, 768]
        b2_ctc_b = b2_model.lm_head.bias.data.clone()    # [32]

        # 2 multi-layer CTC heads
        self.ctc_heads = nn.ModuleList([
            CTCHead(hidden, vocab, intermediate=256)
            for _ in range(NUM_GROUPS)
        ])
        for head in self.ctc_heads:
            # Residual from B2 — strong dysarthric baseline
            head.residual.weight.data.copy_(b2_ctc_w)
            head.residual.bias.data.copy_(b2_ctc_b)
            # net layers: small init so net(x) ≈ 0 at start
            nn.init.normal_(head.net[0].weight, std=0.01)
            nn.init.zeros_(head.net[0].bias)
            nn.init.normal_(head.net[3].weight, std=0.01)
            nn.init.zeros_(head.net[3].bias)

        self.pad_token_id  = b2_model.config.pad_token_id
        self._active_group = 0
        self._load_group(0)

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        backbone_t = sum(p.numel() for p in self.encoder.parameters()
                         if p.requires_grad)
        lora_t    = sum(p.numel() for pl in [self.adapter_A, self.adapter_B]
                        for pg in pl for p in pg)
        head_t    = sum(p.numel() for p in self.ctc_heads.parameters())
        print(f"Total parameters:     {total:,}")
        print(f"Trainable parameters: {trainable:,}")
        print(f"  Backbone top-4:     {backbone_t:,}  (lr=1e-6)")
        print(f"  LoRA adapters:      {lora_t:,}  (lr=1e-4)")
        print(f"  CTC heads (x2):     {head_t:,}  (lr=1e-4)")

    def _load_group(self, g):
        for i, l in enumerate(self._lora_layers):
            l.lora_A.data.copy_(self.adapter_A[g][i].data)
            l.lora_B.data.copy_(self.adapter_B[g][i].data)
        self._active_group = g

    def _save_group(self, g):
        for i, l in enumerate(self._lora_layers):
            self.adapter_A[g][i].data.copy_(l.lora_A.data)
            self.adapter_B[g][i].data.copy_(l.lora_B.data)

    def set_active_group(self, g):
        if g == self._active_group: return
        self._save_group(self._active_group)
        self._load_group(g)

    def forward_single(self, iv, am, g):
        self.set_active_group(g)
        out    = self.encoder(iv, attention_mask=am, output_hidden_states=True)
        logits = self.ctc_heads[g](out.last_hidden_state)
        return logits, out.hidden_states


print("Building model...")
b2_model   = b2_model.cpu()
lora_model = TwoGroupLoRAModel(b2_model).to(device)
b2_model   = b2_model.to(device)
print("Model ready.")

Building model...
  Top 4 backbone layers unfrozen (layers 8-11)
Injecting LoRA...
  Replaced 24 linear layers
  24 LoRALinear layers
Total parameters:     96,693,744
Trainable parameters: 28,300,976
  Backbone top-4:     26,606,640  (lr=1e-6)
  LoRA adapters:      1,234,944  (lr=1e-4)
  CTC heads (x2):     459,392  (lr=1e-4)
Model ready.


## Step 5 — Training helpers

In [9]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def decode_ctc_greedy(pred_ids, tokenizer):
    blank = tokenizer.pad_token_id
    out   = []
    for seq in pred_ids:
        col = [k for k, _ in groupby(seq)]
        fil = [t for t in col if t != blank]
        out.append(tokenizer.decode(fil, skip_special_tokens=True) if fil else "")
    return out


def ctc_loss_fn(logits, labels, pad_id):
    log_p  = F.log_softmax(logits, dim=-1).transpose(0, 1)
    ilen   = torch.full((logits.size(0),), logits.size(1),
                        dtype=torch.long, device=logits.device)
    llen   = (labels != -100).sum(-1)

    # Filter samples where label length >= input length (CTC requirement)
    valid  = llen < ilen
    if not valid.all():
        print(f"  Skipping {(~valid).sum()} samples: label longer than input")
        log_p  = log_p[:, valid]
        ilen   = ilen[valid]
        llen   = llen[valid]
        labels = labels[valid]

    if llen.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=logits.device)

    labs = labels.clone()
    labs[labs == -100] = pad_id
    return F.ctc_loss(log_p, labs, ilen, llen,
                      blank=pad_id, reduction="mean", zero_infinity=True)


def preprocess_batch(samples):
    """Uses data_collator — identical preprocessing to B2 training."""    
    batch  = data_collator(samples)
    iv     = batch["input_values"].to(device)
    am     = batch.get("attention_mask")
    if am is not None: am = am.to(device)
    labels = batch["labels"].to(device)
    return iv, am, labels


def get_layer6_pooled(model, iv, am):
    with torch.no_grad():
        out = model.wavlm(iv, attention_mask=am, output_hidden_states=True)
    l6 = out.hidden_states[7]
    if am is not None:
        fl   = model.wavlm._get_feat_extract_output_lengths(am.sum(-1)).long()
        T    = l6.size(1)
        mask = (torch.arange(T, device=device).unsqueeze(0)
                < fl.unsqueeze(1)).unsqueeze(-1).float()
        return (l6 * mask).sum(1) / mask.sum(1).clamp(min=1)
    return l6.mean(dim=1)


print("Helpers ready.")

Helpers ready.


## Step 6 — Train each group independently

In [14]:
from transformers import get_linear_schedule_with_warmup

LORA_LR     = 1e-5
HEAD_LR     = 1e-5
BACKBONE_LR = 1e-6
NUM_EPOCHS  = 20
PATIENCE    = 5
BATCH_SIZE  = 4
GRAD_ACCUM  = 4


def train_group(grp, train_ds):
    print(f"\n{'='*60}")
    print(f"Training Group {grp} ({GROUP_NAMES[grp]})  [{len(train_ds)} utterances]")
    print(f"{'='*60}")

    lora_model.set_active_group(grp)

    backbone_params = [
        p for p in lora_model.encoder.encoder.layers[8:].parameters()
        if p.requires_grad and type(p).__name__ != "LoRALinear"
    ]

    optimizer = torch.optim.AdamW([
        {"params": list(lora_model.adapter_A[grp].parameters()), "lr": LORA_LR,     "name": "lora"},
        {"params": list(lora_model.adapter_B[grp].parameters()), "lr": LORA_LR,     "name": "lora"},
        {"params": list(lora_model.ctc_heads[grp].parameters()), "lr": HEAD_LR,     "name": "ctc_head"},
        {"params": backbone_params,                               "lr": BACKBONE_LR, "name": "backbone_top4"},
    ], weight_decay=0.01)

    all_trainable = (
        list(lora_model.adapter_A[grp].parameters()) +
        list(lora_model.adapter_B[grp].parameters()) +
        list(lora_model.ctc_heads[grp].parameters()) +
        backbone_params
    )

    # Linear warmup over first 10% of steps
    steps_per_epoch = (len(train_ds) + BATCH_SIZE * GRAD_ACCUM - 1) // (BATCH_SIZE * GRAD_ACCUM)
    total_steps     = steps_per_epoch * NUM_EPOCHS
    warmup_steps    = max(1, int(0.1 * total_steps))
    scheduler       = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    print(f"  Scheduler: linear warmup {warmup_steps} / {total_steps} steps")

    best_loss  = float("inf")
    best_state = None
    no_improve = 0
    indices    = list(range(len(train_ds)))
    n_batches  = (len(indices) + BATCH_SIZE - 1) // BATCH_SIZE

    for epoch in range(NUM_EPOCHS):
        lora_model.train()
        random.shuffle(indices)
        epoch_loss, n_seen = 0.0, 0
        optimizer.zero_grad()

        for bn, i in enumerate(range(0, len(indices), BATCH_SIZE)):
            batch_idx = indices[i:i+BATCH_SIZE]
            samples   = [train_ds[j] for j in batch_idx]
            if not samples: continue

            iv, am, labels = preprocess_batch(samples)
            logits, _      = lora_model.forward_single(iv, am, grp)
            torch.cuda.synchronize()

            loss = ctc_loss_fn(logits, labels, lora_model.pad_token_id)

            if not torch.isfinite(loss):
                print(f"\n  WARNING: non-finite loss at batch {bn} — skipping")
                optimizer.zero_grad()
                continue

            (loss / GRAD_ACCUM).backward()
            epoch_loss += loss.item() * len(samples)
            n_seen     += len(samples)

            if (bn + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            if (bn + 1) % 25 == 0:
                pct = (bn+1)/n_batches
                bar = "█"*int(pct*20) + "░"*(20-int(pct*20))
                print(f"\r  [{bar}] {bn+1}/{n_batches}  "
                      f"loss={epoch_loss/max(n_seen,1):.4f}",
                      end="", flush=True)

        # Final optimizer step for remaining gradients
        torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        print()

        avg_loss = epoch_loss / max(n_seen, 1)
        print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS}  loss={avg_loss:.4f}  "
              f"lr={scheduler.get_last_lr()[0]:.2e}", flush=True)

        if avg_loss < best_loss - 0.005:
            best_loss  = avg_loss
            no_improve = 0
            best_state = {
                "A":        [p.data.clone() for p in lora_model.adapter_A[grp]],
                "B":        [p.data.clone() for p in lora_model.adapter_B[grp]],
                "h":        {k: v.clone() for k, v in
                             lora_model.ctc_heads[grp].state_dict().items()},
                "backbone": [p.data.clone() for p in backbone_params],
            }
            print(f"  → Best: {best_loss:.4f}", flush=True)
        else:
            no_improve += 1
            print(f"  No improvement {no_improve}/{PATIENCE}", flush=True)
            if no_improve >= PATIENCE:
                print("  Early stopping.", flush=True); break

    if best_state:
        for i, p in enumerate(lora_model.adapter_A[grp]):
            p.data.copy_(best_state["A"][i])
        for i, p in enumerate(lora_model.adapter_B[grp]):
            p.data.copy_(best_state["B"][i])
        lora_model.ctc_heads[grp].load_state_dict(best_state["h"])
        for p, saved in zip(backbone_params, best_state["backbone"]):
            p.data.copy_(saved)
        lora_model._load_group(grp)
        print(f"  Restored best (loss={best_loss:.4f})", flush=True)
    return best_loss


train_group(0, group_train_p[0])
train_group(1, group_train_p[1])


Training Group 0 (mild_ctrl)  [1848 utterances]
  Scheduler: linear warmup 232 / 2320 steps
  [███████████████████░] 450/462  loss=0.3122
  Epoch  1/20  loss=0.3095  lr=5.00e-06
  → Best: 0.3095
  [███████████████████░] 450/462  loss=0.3074
  Epoch  2/20  loss=0.3112  lr=1.00e-05
  No improvement 1/5
  [███████████████████░] 450/462  loss=0.3362
  Epoch  3/20  loss=0.3340  lr=9.44e-06
  No improvement 2/5
  [███████████████████░] 450/462  loss=0.3244
  Epoch  4/20  loss=0.3222  lr=8.89e-06
  No improvement 3/5
  [███████████████████░] 450/462  loss=0.2923
  Epoch  5/20  loss=0.2907  lr=8.33e-06
  → Best: 0.2907
  [███████████████████░] 450/462  loss=0.3151
  Epoch  6/20  loss=0.3145  lr=7.78e-06
  No improvement 1/5
  [███████████████████░] 450/462  loss=0.2939
  Epoch  7/20  loss=0.2909  lr=7.22e-06
  No improvement 2/5
  [███████████████████░] 450/462  loss=0.2785
  Epoch  8/20  loss=0.2811  lr=6.67e-06
  → Best: 0.2811
  [███████████████████░] 450/462  loss=0.3113
  Epoch  9/20  lo

0.4979196275193287

In [15]:
torch.save({
    "A0": [p.data for p in lora_model.adapter_A[0]],
    "B0": [p.data for p in lora_model.adapter_B[0]],
    "A1": [p.data for p in lora_model.adapter_A[1]],
    "B1": [p.data for p in lora_model.adapter_B[1]],
    "h0": lora_model.ctc_heads[0].state_dict(),
    "h1": lora_model.ctc_heads[1].state_dict(),
    "backbone_top4": lora_model.encoder.encoder.layers[8:].state_dict(),
}, "/kaggle/working/p1v6_two_group_lora.pt")
print("Model saved.")


Model saved.


## Step 7 — Load Severity MLP and sanity check on test set

In [17]:
class SeverityMLP(nn.Module):
    def __init__(self, hidden=768, n_cls=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128),    nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, n_cls),
        )
    def forward(self, x): return self.net(x)


severity_mlp = SeverityMLP().to(device)
ckpt = torch.load(MLP_PT, map_location="cpu")
severity_mlp.load_state_dict(ckpt["state_dict"] if "state_dict" in ckpt else ckpt)
severity_mlp.eval()
if "best_val_acc" in ckpt:
    print(f"MLP loaded. Val acc at save: {ckpt['best_val_acc']*100:.2f}%")
else:
    print("MLP loaded.")

MLP loaded. Val acc at save: 95.32%


## Step 8 — Collect logits from both groups on test + val sets

In [24]:
def collect_two_group_logits(preprocessed_ds, cats_list, desc=""):
    """
    Run both adapters using preprocessed dataset + data_collator.
    Identical feature extraction to B2 training — avoids ~6pp WER gap.
    """
    lora_model.eval()
    lg0_list, lg1_list, gw_list, lab_list, cat_list, greedy_list = [], [], [], [], [], []

    for i in range(0, len(preprocessed_ds), BATCH):
        end     = min(i + BATCH, len(preprocessed_ds))
        samples = [preprocessed_ds[j] for j in range(i, end)]

        batch = data_collator(samples)
        iv    = batch["input_values"].to(device)
        am    = batch.get("attention_mask")
        if am is not None: am = am.to(device)

        with torch.no_grad():
            lg0, _ = lora_model.forward_single(iv, am, 0)
            lg1, _ = lora_model.forward_single(iv, am, 1)
            pooled = get_layer6_pooled(b2_model, iv, am)
            sp     = F.softmax(severity_mlp(pooled), dim=-1).cpu().numpy()

        # Labels from collated batch
        lab_ids = batch["labels"].numpy()
        lab_ids = np.where(lab_ids != -100, lab_ids, processor.tokenizer.pad_token_id)
        lab_str = processor.tokenizer.batch_decode(lab_ids, skip_special_tokens=True)

        for j in range(len(samples)):
            w0 = float(sp[j][0] + sp[j][1])
            w1 = float(sp[j][2] + sp[j][3])
            lg0_list.append(lg0[j].cpu().numpy())
            lg1_list.append(lg1[j].cpu().numpy())
            gw_list.append([w0, w1])
            lab_list.append(lab_str[j].strip())
            cat_list.append(cats_list[i + j])

            # Greedy decode from blended logits
            blended = w0 * lg0[j].cpu() + w1 * lg1[j].cpu()
            pid     = blended.argmax(dim=-1).numpy()
            text    = decode_ctc_greedy([pid], processor.tokenizer)[0]
            greedy_list.append(text.strip())

        if (i // BATCH + 1) % 20 == 0:
            print(f"  [{desc}] {end}/{len(preprocessed_ds)}")

    return lg0_list, lg1_list, gw_list, lab_list, cat_list, greedy_list


print("Collecting test logits (preprocessed)...")
(all_lg0, all_lg1, all_gw,
 all_labels, all_cats, greedy_preds) = collect_two_group_logits(
    test_p, test_cats_filtered, "test"
)

print("\nCollecting val logits (preprocessed)...")
(val_lg0, val_lg1, val_gw,
 val_labels, val_cats, _) = collect_two_group_logits(
    torgo_val_p, val_cats_filtered, "val"
)

# Greedy baseline on test
eg     = [p if p else "⟨empty⟩" for p in greedy_preds]
gw_wer = wer_metric.compute(predictions=eg, references=all_labels)
gw_cer = cer_metric.compute(predictions=eg, references=all_labels)
res_df = pd.DataFrame({"prediction": greedy_preds,
                        "reference":  all_labels, "Category": all_cats})

print(f"\nP1v6 Greedy (soft-routed): WER={gw_wer*100:.2f}%  CER={gw_cer*100:.2f}%")
for cat in ["control", "mild", "moderate", "severe"]:
    sub = res_df[res_df["Category"] == cat]
    sub = sub[sub["reference"].str.strip().str.len() > 0]
    if not len(sub): continue
    pg  = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
    gw  = wer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    gc  = cer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    print(f"  {cat:10s}: WER={gw*100:.2f}%  CER={gc*100:.2f}%  (n={len(sub)})")


  [test] 160/1786
  [test] 320/1786
  [test] 480/1786
  [test] 640/1786
  [test] 800/1786
  [test] 960/1786
  [test] 1120/1786
  [test] 1280/1786
  [test] 1440/1786
  [test] 1600/1786
  [test] 1760/1786

  [val] 160/1111
  [val] 320/1111
  [val] 480/1111
  [val] 640/1111
  [val] 800/1111
  [val] 960/1111

P1v6 Greedy (soft-routed): WER=40.16%  CER=19.32%
  control   : WER=14.12%  CER=4.97%  (n=302)
  mild      : WER=18.93%  CER=6.02%  (n=673)
  moderate  : WER=71.43%  CER=37.03%  (n=575)
  severe    : WER=64.72%  CER=38.08%  (n=236)


In [25]:
# ── Oracle routing evaluation ──
print("Oracle routing (true severity labels):")
lora_model.eval()
oracle_preds = []

for i in range(0, len(test_p), BATCH):
    end     = min(i + BATCH, len(test_p))
    samples = [test_p[j] for j in range(i, end)]
    cats    = test_cats_filtered[i:end]

    batch = data_collator(samples)
    iv    = batch["input_values"].to(device)
    am    = batch.get("attention_mask")
    if am is not None: am = am.to(device)

    with torch.no_grad():
        # Use true group label — hard routing to correct adapter
        for j, cat in enumerate(cats):
            true_grp   = SEV_TO_GROUP[cat]
            iv_j       = iv[j:j+1]
            am_j       = am[j:j+1] if am is not None else None
            logits, _  = lora_model.forward_single(iv_j, am_j, true_grp)
            pid        = logits.argmax(dim=-1).cpu().numpy()
            text       = decode_ctc_greedy(pid, processor.tokenizer)[0]
            oracle_preds.append(text.strip() if text.strip() else "⟨empty⟩")

o_wer = wer_metric.compute(
    predictions=oracle_preds, references=all_labels
)
o_cer = cer_metric.compute(
    predictions=oracle_preds, references=all_labels
)
print(f"Oracle WER: {o_wer*100:.2f}%  CER: {o_cer*100:.2f}%")
print(f"vs Soft routing: WER={gw_wer*100:.2f}%")
print(f"Gap (oracle advantage): {(gw_wer - o_wer)*100:.2f}pp")

Oracle routing (true severity labels):
Oracle WER: 40.35%  CER: 19.37%
vs Soft routing: WER=40.16%
Gap (oracle advantage): -0.19pp


## Step 9 — KenLM beam search with soft routing

In [27]:
def collect_with_b3_style(dataset_raw, cats_list, desc=""):
    lora_model.eval()
    lg0_list, lg1_list, gw_list, lab_list, cat_list, greedy_list = [], [], [], [], [], []

    for i in range(0, len(dataset_raw), BATCH):
        end     = min(i + BATCH, len(dataset_raw))
        samples = [dataset_raw[j] for j in range(i, end)]
        samples = [s for s in samples if len(s["audio"]["array"]) <= MAX_AUDIO]
        if not samples: continue

        arrays = [s["audio"]["array"] for s in samples]
        sr     = samples[0]["audio"]["sampling_rate"]
        inputs = processor(arrays, sampling_rate=sr,
                           return_tensors="pt", padding=True)
        iv     = inputs.input_values.to(device)
        am     = inputs.attention_mask.to(device) \
                 if "attention_mask" in inputs else None

        with torch.no_grad():
            lora_model.set_active_group(0)
            out0 = lora_model.encoder(iv, attention_mask=am,
                                      output_hidden_states=True)
            lora_model.set_active_group(1)
            out1 = lora_model.encoder(iv, attention_mask=am,
                                      output_hidden_states=True)
            lg0  = lora_model.ctc_heads[0](out0.last_hidden_state)
            lg1  = lora_model.ctc_heads[1](out1.last_hidden_state)
            pooled = get_layer6_pooled(b2_model, iv, am)
            sp     = F.softmax(severity_mlp(pooled), dim=-1).cpu().numpy()

        for j, s in enumerate(samples):
            w0 = float(sp[j][0] + sp[j][1])
            w1 = float(sp[j][2] + sp[j][3])
            lg0_list.append(lg0[j].cpu().numpy())
            lg1_list.append(lg1[j].cpu().numpy())
            gw_list.append([w0, w1])
            lab_list.append(s["transcription"].lower().strip())
            cat_list.append(s["Category"])
            lg_dom = lg0[j].cpu() if w0 >= w1 else lg1[j].cpu()
            pid    = lg_dom.argmax(dim=-1).numpy()
            text   = decode_ctc_greedy([pid], processor.tokenizer)[0]
            greedy_list.append(text.strip())

        if (i // BATCH + 1) % 20 == 0:
            print(f"  [{desc}] {end}/{len(dataset_raw)}")

    return lg0_list, lg1_list, gw_list, lab_list, cat_list, greedy_list


print("Collecting test logits (B3-style raw audio)...")
(all_lg0, all_lg1, all_gw,
 all_labels, all_cats, greedy_preds) = collect_with_b3_style(
    torgo_test, test_cats_filtered, "test"
)

print("\nCollecting val logits (B3-style raw audio)...")
(val_lg0, val_lg1, val_gw,
 val_labels, val_cats, _) = collect_with_b3_style(
    torgo_val, val_cats_filtered, "val"
)

eg     = [p if p else "⟨empty⟩" for p in greedy_preds]
gw_wer = wer_metric.compute(predictions=eg, references=all_labels)
gw_cer = cer_metric.compute(predictions=eg, references=all_labels)
res_df = pd.DataFrame({"prediction": greedy_preds,
                        "reference":  all_labels, "Category": all_cats})

print(f"\nP1v6 Greedy (raw audio): WER={gw_wer*100:.2f}%  CER={gw_cer*100:.2f}%")
for cat in ["control", "mild", "moderate", "severe"]:
    sub = res_df[res_df["Category"] == cat]
    sub = sub[sub["reference"].str.strip().str.len() > 0]
    if not len(sub): continue
    pg  = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
    gw  = wer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    gc  = cer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    print(f"  {cat:10s}: WER={gw*100:.2f}%  CER={gc*100:.2f}%  (n={len(sub)})")

  [test] 160/1786
  [test] 320/1786
  [test] 480/1786
  [test] 640/1786
  [test] 800/1786
  [test] 960/1786
  [test] 1120/1786
  [test] 1280/1786
  [test] 1440/1786
  [test] 1600/1786
  [test] 1760/1786

  [val] 160/1111
  [val] 320/1111
  [val] 480/1111
  [val] 640/1111
  [val] 800/1111
  [val] 960/1111

P1v6 Greedy (raw audio): WER=47.25%  CER=21.43%
  control   : WER=23.38%  CER=7.17%  (n=302)
  mild      : WER=29.25%  CER=8.88%  (n=673)
  moderate  : WER=74.08%  CER=38.48%  (n=575)
  severe    : WER=69.50%  CER=39.42%  (n=236)


In [28]:
lm_dir = "/kaggle/temp/kenlm"
os.makedirs(lm_dir, exist_ok=True)
lm_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/lm.binary", cache_dir=lm_dir,
)
uni_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/unigrams.txt", cache_dir=lm_dir,
)
unigrams = open(uni_path).read().strip().split("\n")
print(f"KenLM ready. Unigrams: {len(unigrams):,}")

# vocab_list at full size 32 — must match logit dimension [T, 32]
vocab_dict = processor.tokenizer.get_vocab()
vocab_list = [None] * len(vocab_dict)  # 32
for tok, idx in vocab_dict.items():
    vocab_list[idx] = tok
vocab_list[processor.tokenizer.pad_token_id] = ""
print(f"vocab_list size: {len(vocab_list)}")  # must be 32

_dec_cache = {}

def get_decoder(alpha, beta):
    key = (round(float(alpha), 3), round(float(beta), 3))
    if key not in _dec_cache:
        _dec_cache[key] = build_ctcdecoder(
            labels=vocab_list,
            kenlm_model_path=lm_path,
            unigrams=unigrams,
            alpha=key[0],
            beta=key[1],
        )
    return _dec_cache[key]


def to_log_probs(lg):
    p = np.exp(lg) / np.exp(lg).sum(axis=-1, keepdims=True)
    return np.log(np.clip(p, 1e-8, 1.0))


def decode_hard(lg0, lg1, gw, alpha, beta, beam_width=100):
    """Hard routing — dominant adapter only. No blending."""
    lg = lg0 if gw[0] >= gw[1] else lg1
    return get_decoder(alpha, beta).decode(
        to_log_probs(lg), beam_width=beam_width
    ).strip().lower()


def decode_best_beam(lg0, lg1, gw, alpha, beta, beam_width=100, n_best=10):
    """Both adapters beam search independently, MLP-weighted selection."""
    dec    = get_decoder(alpha, beta)
    beams0 = dec.decode_beams(to_log_probs(lg0), beam_width=beam_width)[:n_best]
    beams1 = dec.decode_beams(to_log_probs(lg1), beam_width=beam_width)[:n_best]
    text0  = beams0[0][0] if beams0 else ""
    text1  = beams1[0][0] if beams1 else ""
    score0 = (beams0[0][3] + beams0[0][4]) if beams0 else -1e9
    score1 = (beams1[0][3] + beams1[0][4]) if beams1 else -1e9
    return (text0 if gw[0]*score0 >= gw[1]*score1 else text1).strip().lower()


def decode_oracle(lg0, lg1, true_cat, alpha, beta, beam_width=100):
    """Oracle routing — true severity label. Upper bound."""
    lg = lg0 if SEV_TO_GROUP[true_cat] == 0 else lg1
    return get_decoder(alpha, beta).decode(
        to_log_probs(lg), beam_width=beam_width
    ).strip().lower()


print("Decoder helpers ready.")

language_model/lm.binary:   0%|          | 0.00/863M [00:00<?, ?B/s]

language_model/unigrams.txt:   0%|          | 0.00/3.51M [00:00<?, ?B/s]

KenLM ready. Unigrams: 373,978
vocab_list size: 32
Decoder helpers ready.


In [29]:
best_beta = 1.989
best_alpha = 0.459
BEAM_WIDTH = 100

print(f"Final eval: alpha={best_alpha}  beta={best_beta}  beam={BEAM_WIDTH}")
print("=" * 65)

strategies = [
    ("hard_routing",   lambda i: decode_hard(
        all_lg0[i], all_lg1[i], all_gw[i], best_alpha, best_beta, BEAM_WIDTH)),
    ("best_beam",      lambda i: decode_best_beam(
        all_lg0[i], all_lg1[i], all_gw[i], best_alpha, best_beta, BEAM_WIDTH)),
    ("oracle_routing", lambda i: decode_oracle(
        all_lg0[i], all_lg1[i], all_cats[i], best_alpha, best_beta, BEAM_WIDTH)),
]

for name, decode_fn in strategies:
    print(f"\nRunning {name}...")
    preds = [decode_fn(i) for i in range(len(all_lg0))]
    ep    = [p if p else "⟨empty⟩" for p in preds]
    wer   = wer_metric.compute(predictions=ep, references=all_labels)
    cer   = cer_metric.compute(predictions=ep, references=all_labels)
    df_   = pd.DataFrame({"prediction": preds,
                           "reference":  all_labels, "Category": all_cats})
    print(f"  WER={wer*100:.2f}%  CER={cer*100:.2f}%  "
          f"Improvement={(gw_wer-wer)*100:.2f}pp  vs B3={(wer-0.3236)*100:+.2f}pp")
    for cat in ["control", "mild", "moderate", "severe"]:
        sub = df_[df_["Category"] == cat]
        sub = sub[sub["reference"].str.strip().str.len() > 0]
        if not len(sub): continue
        pf  = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
        fw  = wer_metric.compute(predictions=pf, references=sub["reference"].tolist())
        fc  = cer_metric.compute(predictions=pf, references=sub["reference"].tolist())
        print(f"    {cat:10s}: WER={fw*100:.2f}%  CER={fc*100:.2f}%")
    if name == "best_beam":
        df_.to_csv("/kaggle/working/p1v6_results.csv", index=False)

print("\n" + "=" * 65)
print("Summary")
print("=" * 65)
print(f"{'System':25s} {'WER':>8} {'vs B3':>10}")
print("-" * 46)
print(f"{'B2 Greedy':25s} {'50.53%':>8}")
print(f"{'B3 (B2+KenLM)':25s} {'32.36%':>8}  {'—':>10}")
print(f"{'P1v6 Greedy':25s} {gw_wer*100:>7.2f}%")

with open("/kaggle/working/p1v6_summary.json", "w") as f:
    json.dump({
        "model":      "P1v6_2Group_LoRA_KenLM",
        "processing": "B3-style raw audio",
        "alpha":      best_alpha,
        "beta":       best_beta,
        "greedy_wer": round(gw_wer, 4),
        "b3_wer":     0.3236,
    }, f, indent=2)
print("Results saved.")

Final eval: alpha=0.459  beta=1.989  beam=100

Running hard_routing...
  WER=29.43%  CER=16.54%  Improvement=17.82pp  vs B3=-2.93pp
    control   : WER=10.15%  CER=4.11%
    mild      : WER=9.73%  CER=4.30%
    moderate  : WER=55.43%  CER=32.01%
    severe    : WER=53.19%  CER=35.01%

Running best_beam...
  WER=29.40%  CER=16.52%  Improvement=17.84pp  vs B3=-2.96pp
    control   : WER=9.71%  CER=4.02%
    mild      : WER=9.91%  CER=4.38%
    moderate  : WER=55.43%  CER=31.95%
    severe    : WER=53.01%  CER=34.87%

Running oracle_routing...
  WER=29.31%  CER=16.50%  Improvement=17.94pp  vs B3=-3.05pp
    control   : WER=9.85%  CER=4.02%
    mild      : WER=9.73%  CER=4.31%
    moderate  : WER=54.96%  CER=31.84%
    severe    : WER=53.72%  CER=35.20%

Summary
System                         WER      vs B3
----------------------------------------------
B2 Greedy                   50.53%
B3 (B2+KenLM)               32.36%           —
P1v6 Greedy                 47.25%
Results saved.
